# Phase 3: preparing the images

**CRISP-DM stage:** Data preparation

**Runs on:** this first part runs on the laptop's processor, on 2 photos only, because it only demonstrates settings. No model runs. The laptop's PyTorch version differs from Kaggle's, so this is a demonstration, not a reported result. The real prepared batch is checked on Kaggle later.

**Figures with dataset images:** files named `gallery_*` show real parts, so they stay on the laptop and out of git. Each figure is saved and closed, not displayed, so no image ends up in the notebook.

## 1. Find the photos

Both photos are defective, in `regular` lighting, and already in the laptop sample. The code checks that each is in half A, because half B stays sealed.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from anomalib.data.utils import read_image, read_mask
from anomalib.models import Padim, Patchcore, ReverseDistillation, WinClip
from torchvision.tv_tensors import Image as TvImage
from torchvision.tv_tensors import Mask as TvMask

# Lets the notebook run from the project folder or from the notebooks folder
PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent

SAMPLE_FOLDER = PROJECT_FOLDER / "data" / "sample"
RESULTS_FOLDER = PROJECT_FOLDER / "outputs" / "results" / "03_preprocessing"
FIGURES_FOLDER = PROJECT_FOLDER / "outputs" / "figures"
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

PHOTOS = {"sheet_metal": "000_regular.png", "vial": "001_regular.png"}
FIGURE_DPI = 150

half_a_names = []
split_file = PROJECT_FOLDER / "outputs" / "results" / "02_exploration" / "half_a_files.csv"
with open(split_file, newline="") as list_file:
    for row in csv.DictReader(list_file):
        if row["label"] == "bad":
            half_a_names.append((row["product"], row["file_name"]))

for product_name in PHOTOS:
    if (product_name, PHOTOS[product_name]) not in half_a_names:
        raise ValueError("This photo is not in half A: " + PHOTOS[product_name])
print("Both photos are in half A")

## 2. anomalib's default recipes

Each method has a `configure_pre_processor` function, and calling it with no arguments gives its default recipe. This prints the real settings, so nothing is guessed.

- **`Resize`** changes the photo to a fixed size. The size is given as `[height, width]`, so a non-square photo gets squashed. `antialias=True` smooths pixels when shrinking, so fine patterns do not break up.
- **`interpolation`** is how new pixels are worked out from old ones. Bilinear blends the nearest 4 pixels, and bicubic blends the nearest 16.
- **`Normalize`** changes the pixel numbers, explained in part 3.
- **There is no crop** in these default recipes. PatchCore only crops if `center_crop_size` is passed.

In [ ]:
result_lines = []

METHODS = [
    ("PaDiM", Padim),
    ("PatchCore", Patchcore),
    ("Reverse Distillation", ReverseDistillation),
    ("WinCLIP", WinClip),
]

for method_name, method in METHODS:
    result_lines.append("=== " + method_name + " default recipe")
    result_lines.append(str(method.configure_pre_processor().transform))

for line in result_lines:
    print(line)

## 3. Normalization, checked by hand

For each colour channel, normalization does this sum:

`new value = (old value - mean) / std`

The mean is the average brightness of the network's training photos, and std is how much that brightness typically varies. This re-centres a photo's numbers onto the scale the network learned on.

The small test normalizes one made-up pixel with anomalib's own step, then with the sum by hand. The two must match.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TOLERANCE = 0.00001


def normalize_by_hand(value, mean, std):
    """Do the normalization sum for one number."""
    return (value - mean) / std


# The recipe is a list of steps. Position 1 is its Normalize step
imagenet_recipe = Padim.configure_pre_processor().transform
normalize_step = imagenet_recipe.transforms[1]

# A made-up single pixel with three channels, each 0.5
test_pixel = torch.full((1, 3, 1, 1), 0.5)
test_result = normalize_step(test_pixel)

for channel in range(3):
    by_hand = normalize_by_hand(0.5, IMAGENET_MEAN[channel], IMAGENET_STD[channel])
    if abs(float(test_result[0, channel, 0, 0]) - by_hand) > TOLERANCE:
        raise ValueError("anomalib's normalization differs from the sum by hand")
print("Normalization test passed:", test_result.flatten().tolist())

## 4. Before and after, on real photos

`TvImage` and `TvMask` label which is the photo and which is the mask. The resize step uses this label to treat them differently, and the printed mask values below show whether the mask stays pure 0 and 1.

The worked pixel is the centre of the resized photo. All three stored channels are equal, because the photos are greyscale, and each becomes a different number because each channel has its own mean and std.

**The figure:** left is the original photo, middle is after the default resize, and right is after normalization. The right panel shows one channel with its brightness stretched, because its numbers no longer fit a normal display scale. Red outlines show the mask.

In [ ]:
RECIPES = [
    ("ImageNet recipe, PaDiM, PatchCore, Reverse Distillation", Padim.configure_pre_processor().transform),
    ("CLIP recipe, WinCLIP", WinClip.configure_pre_processor().transform),
]
FIGURE_SIZES = {"sheet_metal": (18, 4), "vial": (13, 6)}

for product_name in PHOTOS:
    file_name = PHOTOS[product_name]
    test_folder = SAMPLE_FOLDER / product_name / product_name / "test_public"
    mask_name = Path(file_name).stem + "_mask.png"

    image = read_image(test_folder / "bad" / file_name, as_tensor=True)
    mask = read_mask(test_folder / "ground_truth" / "bad" / mask_name, as_tensor=True)

    result_lines.append("")
    result_lines.append("=== " + product_name + ", " + file_name)
    result_lines.append("as loaded: photo " + str(list(image.shape)) + ", values "
                        + str(round(float(image.min()), 3)) + " to " + str(round(float(image.max()), 3)))
    result_lines.append("as loaded: mask " + str(list(mask.shape)) + ", defect pixels " + str(int(mask.sum())))

    for recipe_name, recipe in RECIPES:
        resize_step = recipe.transforms[0]
        normalize_step = recipe.transforms[1]
        # One extra dimension is added, because these steps expect a batch of photos
        resized_image, resized_mask = resize_step(TvImage(image.unsqueeze(0)), TvMask(mask.unsqueeze(0)))
        normalized_image = normalize_step(resized_image)

        result_lines.append(recipe_name + ":")
        result_lines.append("  after resize: photo " + str(list(resized_image.shape[1:])) + ", values "
                            + str(round(float(resized_image.min()), 3)) + " to " + str(round(float(resized_image.max()), 3)))
        result_lines.append("  after resize: mask values " + str(torch.unique(resized_mask.int()).tolist())
                            + ", defect pixels " + str(int(resized_mask.sum())))
        result_lines.append("  after normalize: values " + str(round(float(normalized_image.min()), 3))
                            + " to " + str(round(float(normalized_image.max()), 3)))

        # The worked pixel: the centre of the resized photo
        center_row = resized_image.shape[2] // 2
        center_column = resized_image.shape[3] // 2
        means = normalize_step.mean
        stds = normalize_step.std
        for channel in range(3):
            old_value = float(resized_image[0, channel, center_row, center_column])
            new_value = float(normalized_image[0, channel, center_row, center_column])
            by_hand = normalize_by_hand(old_value, means[channel], stds[channel])
            result_lines.append("  centre pixel, channel " + str(channel) + ": (" + str(round(old_value, 4)) + " - " + str(round(means[channel], 4))
                                + ") / " + str(round(stds[channel], 4)) + " = " + str(round(by_hand, 4))
                                + ", anomalib gave " + str(round(new_value, 4)))

        # The figure uses the recipe shared by three of the four methods
        if recipe_name.startswith("ImageNet"):
            figure, axes = plt.subplots(1, 3, figsize=FIGURE_SIZES[product_name])

            axes[0].imshow(image[0], cmap="gray", vmin=0, vmax=1)
            axes[0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
            axes[0].set_title("Original, " + str(image.shape[2]) + " x " + str(image.shape[1]), fontsize=10)

            axes[1].imshow(resized_image[0, 0], cmap="gray", vmin=0, vmax=1)
            axes[1].contour(resized_mask[0], levels=[0.5], colors="red", linewidths=0.8)
            axes[1].set_title("After default resize, " + str(resized_image.shape[3]) + " x " + str(resized_image.shape[2]), fontsize=10)

            # matplotlib's default stretches this panel from its own lowest to highest value
            axes[2].imshow(normalized_image[0, 0], cmap="gray")
            axes[2].set_title("After normalization, one channel, stretched", fontsize=10)

            for axis in axes:
                axis.axis("off")
            figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A photo. Red outline: mask")
            figure.savefig(FIGURES_FOLDER / ("gallery_preprocessing_" + product_name + ".png"), dpi=FIGURE_DPI, bbox_inches="tight")
            # Closing without showing keeps dataset images out of the notebook
            plt.close(figure)

# Numbers only, with no image content, so this file can be committed
with open(RESULTS_FOLDER / "preprocessing_defaults.txt", "w") as summary_file:
    for line in result_lines:
        summary_file.write(line + "\n")

for line in result_lines:
    print(line)

## 5. Are masks resized with nearest-neighbour?

A mask says defect or no defect, with nothing in between. **Nearest-neighbour** gives each new pixel the value of the closest old pixel, so a mask stays pure 0 and 1. **Bilinear**, used for photos, blends neighbouring pixels, which would create values such as 0.5, meaning "half a defect", which has no meaning.

In torchvision, a mask labelled `TvMask` only ever uses nearest-neighbour. This confirms that on real masks at sizes 256 and 512, by comparing with a separate nearest-neighbour calculation. Bilinear is shown only for comparison.

The small test uses a made-up mask with a known answer.

In [ ]:
from torch.nn.functional import interpolate

MAIN_SIZES = [256, 512]

# Made-up 4 by 4 mask, left half defect. Known answer at 2 by 2: left column 1, right column 0
test_mask = torch.tensor([[1, 1, 0, 0]] * 4, dtype=torch.uint8).unsqueeze(0)
test_resize = Padim.configure_pre_processor(image_size=(2, 2)).transform.transforms[0]
test_result = test_resize(TvMask(test_mask))
if test_result.tolist() != [[[1, 0], [1, 0]]]:
    raise ValueError("Mask resize test failed: " + str(test_result.tolist()))
print("Mask resize test passed")

mask_lines = []

for product_name in PHOTOS:
    test_folder = SAMPLE_FOLDER / product_name / product_name / "test_public"
    mask_name = Path(PHOTOS[product_name]).stem + "_mask.png"
    mask = read_mask(test_folder / "ground_truth" / "bad" / mask_name, as_tensor=True)
    # interpolate expects extra dimensions for batch and channel
    mask_as_decimals = mask.float().unsqueeze(0).unsqueeze(0)

    for size in MAIN_SIZES:
        resize_step = Padim.configure_pre_processor(image_size=(size, size)).transform.transforms[0]
        resized = resize_step(TvMask(mask.unsqueeze(0)))[0]
        separate = interpolate(mask_as_decimals, size=(size, size), mode="nearest")[0, 0]
        bilinear = interpolate(mask_as_decimals, size=(size, size), mode="bilinear", align_corners=False)[0, 0]
        in_between = ((bilinear > 0) & (bilinear < 1)).sum()

        mask_lines.append("=== " + product_name + ", " + PHOTOS[product_name] + ", size " + str(size))
        mask_lines.append("recipe's values: " + str(torch.unique(resized.int()).tolist()))
        mask_lines.append("matches separate nearest-neighbour: " + str(bool((resized.float() == separate).all())))
        mask_lines.append("defect pixels: " + str(int(resized.sum())) + " of " + str(size * size))
        mask_lines.append("bilinear would give in-between values at: " + str(int(in_between)) + " pixels")

with open(RESULTS_FOLDER / "mask_check.txt", "w") as summary_file:
    for line in mask_lines:
        summary_file.write(line + "\n")

for line in mask_lines:
    print(line)

## 6. Do blank borders change the score?

Padded pixels count as good pixels, and they are easy to get right. AU-PRO limits false alarms as a share of good pixels, so extra easy pixels could make the limit easier to meet.

This small test uses made-up scores, with anomalib's own AU-PRO at a 30 percent limit, which is one of the benchmark's limits. It compares the same scores with and without added border pixels, which are given the lowest score, as if the method ignored them. 3 extra columns for each original one matches Sheet Metal, whose padded photo fills 25 percent of the square.

**Limit:** made-up scores show whether the effect can happen, not how large it would be for a real method. Real border pixels may not all get the lowest score.

In [ ]:
from anomalib.metrics.aupro import _AUPRO

MADE_UP_SIZE = 64
BORDER_WIDTH = 3 * MADE_UP_SIZE
FALSE_POSITIVE_LIMIT = 0.3

generator = torch.Generator().manual_seed(42)
made_up_scores = torch.rand(1, MADE_UP_SIZE, MADE_UP_SIZE, generator=generator)
made_up_mask = torch.zeros(1, MADE_UP_SIZE, MADE_UP_SIZE, dtype=torch.int64)
made_up_mask[0, 20:30, 20:30] = 1
# Defect pixels score a little higher on average, so the method is imperfect but useful
made_up_scores = made_up_scores + 0.3 * made_up_mask

border_scores = torch.full((1, MADE_UP_SIZE, BORDER_WIDTH), float(made_up_scores.min()))
border_mask = torch.zeros(1, MADE_UP_SIZE, BORDER_WIDTH, dtype=torch.int64)
padded_scores = torch.cat([made_up_scores, border_scores], dim=2)
padded_mask = torch.cat([made_up_mask, border_mask], dim=2)

metric = _AUPRO(fpr_limit=FALSE_POSITIVE_LIMIT)
metric.update(made_up_scores, made_up_mask)
score_without_border = float(metric.compute())

metric = _AUPRO(fpr_limit=FALSE_POSITIVE_LIMIT)
metric.update(padded_scores, padded_mask)
score_with_border = float(metric.compute())

score_lines = []
score_lines.append("made-up test, AU-PRO at a 30 percent limit")
score_lines.append("without border: " + str(round(score_without_border, 4)))
score_lines.append("with border pixels added: " + str(round(score_with_border, 4)))
for line in score_lines:
    print(line)

## 7. The options side by side

Three options, shown with the mask outlined:

- **Squash:** anomalib's default resize. Nothing is cut off, but shapes are distorted.
- **Pad:** anomalib's `SquarePad`, then resize. Shapes stay true. By default it repeats the photo's edge pixels, so borders are not plain black, while the mask's border is filled with 0, meaning no defect.
- **Crop, square from the centre:** keeps a square as wide as the shorter side, then resize. Kept parts stay large and undistorted, but anything outside is gone.

**A trap with `SquarePad`:** it returns plain numbers without the photo and mask labels, so the next resize step would shrink only the photo. It also repeats edge pixels unless the mask is stored as true or false, which would copy defects into the border. So `pad_photo_and_mask` pads the mask as true or false, then puts the labels back. A made-up mask with a known answer checks this.

The table uses the half A measurements from Phase 2. Sizes are arithmetic, and the crop's losses come from `defect_location_summary.txt`.

In [ ]:
from anomalib.data.transforms import SquarePad
from torchvision.transforms.v2 import CenterCrop

defect_rows = []
sizes_file = PROJECT_FOLDER / "outputs" / "results" / "02_exploration" / "defect_sizes.csv"
with open(sizes_file, newline="") as list_file:
    for row in csv.DictReader(list_file):
        defect_rows.append(row)

PHOTO_SIZES = {"sheet_metal": (4224, 1056), "vial": (1400, 1900)}


def pad_photo_and_mask(photo, mask):
    """Pad both to a square, then restore the labels the resize step relies on."""
    padded_photo = SquarePad()(photo)
    # As true or false, the mask's border is filled with no defect, not copied from its edge
    padded_mask = SquarePad()(mask.bool())
    return TvImage(padded_photo), TvMask(padded_mask.to(torch.uint8))


# Made-up 2 by 6 mask with one defect pixel on the top edge. Known answer: still 1 pixel, 6 by 6
test_photo = TvImage(torch.zeros(1, 3, 2, 6))
test_edge_mask = torch.zeros(1, 2, 6, dtype=torch.uint8)
test_edge_mask[0, 0, 2] = 1
ignored_photo, test_padded = pad_photo_and_mask(test_photo, TvMask(test_edge_mask))
if list(test_padded.shape) != [1, 6, 6] or int(test_padded.sum()) != 1:
    raise ValueError("Padding test failed: " + str(test_padded.tolist()))
print("Padding test passed")

option_lines = score_lines + [""]

for product_name in PHOTOS:
    width, height = PHOTO_SIZES[product_name]
    longest = max(width, height)
    shortest = min(width, height)
    product_rows = []
    for row in defect_rows:
        if row["product"] == product_name:
            product_rows.append(row)

    option_lines.append("=== " + product_name + ", half A, " + str(len(product_rows)) + " defects counting each lighting")
    for size in MAIN_SIZES:
        options = [
            ("squash", size / width, size / height),
            ("pad", size / longest, size / longest),
        ]
        for option_name, width_scale, height_scale in options:
            areas = []
            under_one_area = 0
            under_one_side = 0
            for row in product_rows:
                new_area = int(row["area_pixels"]) * width_scale * height_scale
                areas.append(new_area)
                if new_area < 1:
                    under_one_area = under_one_area + 1
                if int(row["width"]) * width_scale < 1 or int(row["height"]) * height_scale < 1:
                    under_one_side = under_one_side + 1
            areas.sort()
            filled = 100 * width * width_scale * height * height_scale / (size * size)
            option_lines.append(str(size) + " " + option_name + ": photo fills " + str(round(filled, 1)) + " percent"
                                + ", middle defect " + str(round(areas[len(areas) // 2], 1)) + " pixels"
                                + ", under 1 pixel in area " + str(under_one_area)
                                + ", under 1 pixel wide or tall " + str(under_one_side))
        kept_scale = size / shortest
        option_lines.append(str(size) + " crop: keeps " + str(shortest) + " x " + str(shortest)
                            + ", one new pixel covers " + str(round(1 / (kept_scale * kept_scale), 1)) + " old pixels")

    # Same recipe steps as the default, with one extra step in front
    test_folder = SAMPLE_FOLDER / product_name / product_name / "test_public"
    file_name = PHOTOS[product_name]
    image = read_image(test_folder / "bad" / file_name, as_tensor=True)
    mask = read_mask(test_folder / "ground_truth" / "bad" / (Path(file_name).stem + "_mask.png"), as_tensor=True)
    tv_image = TvImage(image.unsqueeze(0))
    tv_mask = TvMask(mask.unsqueeze(0))
    resize_step = Padim.configure_pre_processor(image_size=(256, 256)).transform.transforms[0]

    squashed_image, squashed_mask = resize_step(tv_image, tv_mask)
    padded_image, padded_mask_real = resize_step(*pad_photo_and_mask(tv_image, tv_mask))
    cropped_image, cropped_mask = resize_step(*CenterCrop((shortest, shortest))(tv_image, tv_mask))

    panels = [
        ("Squash", squashed_image, squashed_mask),
        ("Pad", padded_image, padded_mask_real),
        ("Crop, square from the centre", cropped_image, cropped_mask),
    ]
    figure, axes = plt.subplots(1, 3, figsize=(13, 5))
    for position in range(3):
        panel_name, panel_image, panel_mask = panels[position]
        axes[position].imshow(panel_image[0, 0], cmap="gray", vmin=0, vmax=1)
        if int(panel_mask.sum()) > 0:
            axes[position].contour(panel_mask[0], levels=[0.5], colors="red", linewidths=0.8)
        axes[position].set_title(panel_name + ", 256 x 256, defect pixels " + str(int(panel_mask.sum())), fontsize=10)
        axes[position].axis("off")
    figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A photo. Red outline: mask")
    figure.savefig(FIGURES_FOLDER / ("gallery_shape_options_" + product_name + ".png"), dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook
    plt.close(figure)
    option_lines.append("this photo at 256, defect pixels: squash " + str(int(squashed_mask.sum()))
                        + ", pad " + str(int(padded_mask_real.sum())) + ", crop " + str(int(cropped_mask.sum())))
    option_lines.append("")

# Numbers only, with no image content, so this file can be committed
with open(RESULTS_FOLDER / "shape_options.txt", "w") as summary_file:
    for line in option_lines:
        summary_file.write(line + "\n")

for line in option_lines:
    print(line)